### 02 - Text - Preprocessing - Pitt

In this notebook, we will pre-process text files in Pitt set and split them into train / dev / test set with text.

It will read split id files (`pitt_train.csv`, `pitt_dev.csv`, `pitt_test.csv`) and map the text accordingly. 
Then it will export files with id, text and labels to `pitt_train_txt.csv`, `pitt_dev_txt.csv`, `pitt_test_txt.csv` for model training

In [1]:
import pandas as pd

In [2]:
# please point to correct downloaded transcribed folders of the original dataset
data_path = "../../../data/" 

fname_train = data_path + "pitt/pitt_train.csv"
fname_dev = data_path + "pitt/pitt_dev.csv"
fname_test = data_path + "pitt/pitt_test.csv"
fname_pitt_text = data_path + "pitt/text_pitt.csv"

In [3]:
train_df = pd.read_csv(fname_train)
dev_df = pd.read_csv(fname_dev)
test_df = pd.read_csv(fname_test)
text_df = pd.read_csv(fname_pitt_text)

In [4]:
# quick look at the dataframes
print("Train len =", len(train_df))
train_df['id'] = train_df['id'].astype(str)
train_df.head()

Train len = 174


,id,target,age_of_first_test,sex,less_than_65
0,714,ad,79,male,False
1,466,ad,78,female,False
2,511,ad,75,female,False
3,609,ad,70,female,False
4,504,ad,78,male,False


In [5]:
print("Dev len =", len(dev_df))
dev_df['id'] = dev_df['id'].astype(str)
dev_df.head()

Dev len = 59


,id,target,age_of_first_test,sex,less_than_65
0,234,ad,66,male,False
1,256,cn,67,male,False
2,497,ad,70,female,False
3,264,ad,65,female,False
4,297,cn,64,male,True


In [6]:
print("Test len =", len(test_df))
test_df['id'] = test_df['id'].astype(str)
test_df.head()

Test len = 59


,id,target,age_of_first_test,sex,less_than_65
0,358,ad,60,female,True
1,34,cn,70,male,False
2,42,cn,56,female,True
3,650,ad,87,female,False
4,332,cn,50,male,True


In [7]:
print("Pitt text len =", len(text_df))
# map file_id to id
text_df["id"] = text_df["file_id"].apply(lambda fid: str(fid).split("-")[0].strip()) # e.g. 001-0 -> 001
text_df.head()

Pitt text len = 549


,file_id,text,target,id
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,001
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,001
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,003
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,005
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,005


In [8]:
def get_split_name(id):
    """
        return split name (train/dev/set) given id
    """
    result = "train" # default
    id = str(id)
    if len(dev_df.query("id == @id")) > 0:
        result = "dev"
    elif len(test_df.query("id == @id")) > 0:
        result = "test"
        
    return result

assert get_split_name("714") == "train"
assert get_split_name("466") == "train"
assert get_split_name("511") == "train"

assert get_split_name("234") == "dev"
assert get_split_name("256") == "dev"
assert get_split_name("497") == "dev"


assert get_split_name("358") == "test"
assert get_split_name("34") == "test"
assert get_split_name("332") == "test"

print("Passed")

Passed


In [9]:
# apply to text df
text_df["split_name"] = text_df["id"].apply(lambda id: get_split_name(id))
text_df.head()

,file_id,text,target,id,split_name
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,001,train
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,001,train
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,003,train
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,005,train
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,005,train


In [10]:
print(text_df.query("id == '001'"))

  file_id                                               text target   id  \
0   001-0  mhm .\nalright .\nthere's &-um a young boy tha...     AD  001   
1   001-2  mhm .\nthere's a young boy &-uh going in a coo...     AD  001   

  split_name  
0      train  
1      train  


In [11]:
assert text_df.query("id == '609'")["split_name"].iloc[0] == "train"
assert text_df.query("id == '504'")["split_name"].iloc[0] == "train"


assert text_df.query("id == '264'")["split_name"].iloc[0] == "dev"
assert text_df.query("id == '297'")["split_name"].iloc[0] == "dev"

assert text_df.query("id == '650'")["split_name"].iloc[0] == "test"
assert text_df.query("id == '332'")["split_name"].iloc[0] == "test"
print("Passed")

Passed


In [12]:
# split to 3 files
train_fname_out = data_path + "pitt/pitt_train_txt.csv"
dev_fname_out = data_path + "pitt/pitt_dev_txt.csv"
test_fname_out = data_path + "pitt/pitt_test_txt.csv"

txt_train_df = text_df.query("split_name == 'train'")
txt_dev_df = text_df.query("split_name == 'dev'")
txt_test_df = text_df.query("split_name == 'test'")

In [13]:
# more assert test to make sure no id leakage between train / dev / test set
train_ids = set(txt_train_df["id"].tolist())
dev_ids = set(txt_dev_df["id"].tolist())
test_ids = set(txt_test_df["id"].tolist())


assert len(train_ids.intersection(dev_ids)) == 0
assert len(train_ids.intersection(test_ids)) == 0
assert len(dev_ids.intersection(test_ids)) == 0

print("Passed")

Passed


In [14]:
txt_train_df.head()

,file_id,text,target,id,split_name
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,001,train
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,001,train
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD,003,train
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,005,train
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,005,train


In [15]:
txt_dev_df.head()

,file_id,text,target,id,split_name
81,122-0,"well, why <the mother is> &-um <she's not wit...",AD,122,dev
82,122-1,I see a tad bit .\nand somebody is doin(g) dr...,AD,122,dev
83,125-0,cookie jar .\ncookie jar .\n(..) plate .\n(..)...,AD,125,dev
85,134-0,well ‡ there's a boy trying to get some cookie...,AD,134,dev
86,134-1,a little boy is <tryin(g) to get the> &-uh ge...,AD,134,dev


In [16]:
txt_test_df.head()

,file_id,text,target,id,split_name
84,127-0,(..) well ‡ the mother's washing the dishes .\...,AD,127,test
108,181-0,well ‡ they they look like they're stealin(g)...,AD,181,test
109,181-1,a little boy is gettin(g) himself some cookies...,AD,181,test
110,181-2,uh she's &-um &+w &+w &+w washing dishes .\nan...,AD,181,test
111,181-3,uh &+th these two little &+g kids are gettin(g...,AD,181,test


In [17]:
# write to csv files
print("Writing to csv files...")
txt_train_df.to_csv(train_fname_out,
                    index=False
                   )
txt_dev_df.to_csv(dev_fname_out,
                  index=False
                 )
txt_test_df.to_csv(test_fname_out,
                   index=False
                   )

print("Completed")

Writing to csv files...
Completed
